In [1]:
%cd /content
!rm -rf fast-failure-detection
!git clone https://github.com/jieyingc/fast-failure-detection.git

/content
Cloning into 'fast-failure-detection'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (129/129), done.
remote: Total 149 (delta 70), reused 40 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 817.74 KiB | 3.31 MiB/s, done.
Resolving deltas: 100% (70/70), done.


In [2]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/content/fast-failure-detection")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import run_cv
from src.config import LOG_FEATURES, MODEL_NAME

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("/content/drive/MyDrive/fast-failure-detection/results/tool_features_full.csv")

baseline_df = pd.read_csv(DATA_PATH)
baseline_df = baseline_df[baseline_df["patch_is_empty"] == 0].copy()

print(baseline_df.shape)
baseline_df.head()

(18156, 45)


,instance_id,traj_id,repo_id,model,resolved,patch_is_empty,patch_char_len,num_lines_added,num_lines_removed,num_files_touched,...,user_text_clean,assistant_text_clean,assistant_content_text_clean,tool_text_clean,tool_observation_text_clean,file_path_text_clean,patch_text_clean,contains_assert_change,contains_import_change,contains_exception_handling_change
1,marshmallow-code__apispec.8b421526.func_pm_rem...,marshmallow-code__apispec.8b421526.func_pm_rem...,marshmallow-code__apispec,claude-3-7-sonnet-20250219,1,0,1925,15,12,1,...,<uploaded_files>\n/testbed\n</uploaded_files>\...,i'll help you implement the necessary changes ...,i'll help you implement the necessary changes ...,observation:\n/testbed/docs/conf.py\r\n/testbe...,observation:\n/testbed/docs/conf.py\r\n/testbe...,flashtext keyword py,for key in current_dict if key _keyword_ terms...,0,0,0
3,marshmallow-code__marshmallow.9716fc62.func_pm...,marshmallow-code__marshmallow.9716fc62.func_pm...,marshmallow-code__marshmallow,claude-3-7-sonnet-20250219,1,0,14794,4,261,3,...,<uploaded_files>\n/testbed\n</uploaded_files>\...,i'll help you implement the necessary changes ...,i'll help you implement the necessary changes ...,observation:\n/testbed/docs/conf.py\r\n/testbe...,observation:\n/testbed/docs/conf.py\r\n/testbe...,license txt setup py textfsm parser py,apache license version 2 0 january 2004 http w...,1,1,1
6,sloria__environs.73c372df.func_pm_remove_assig...,sloria__environs.73c372df.func_pm_remove_assig...,sloria__environs,claude-3-7-sonnet-20250219,1,0,575,1,1,1,...,<uploaded_files>\n/testbed\n</uploaded_files>\...,i'll help you implement the necessary changes ...,i'll help you implement the necessary changes ...,observation:\n/testbed/examples/deferred_valid...,observation:\n/testbed/examples/deferred_valid...,gpxpy geo py,d abs a c d abs a point latitude b point longi...,0,0,0
7,amueller__word_cloud.ec24191c.func_pm_ctrl_shu...,amueller__word_cloud.ec24191c.func_pm_ctrl_shu...,amueller__word_cloud,claude-3-7-sonnet-20250219,1,0,574,1,1,1,...,<uploaded_files>\n/testbed\n</uploaded_files>\...,i'll help you implement the necessary changes ...,i'll help you implement the necessary changes ...,observation:\n/testbed/doc/conf.py\r\n/testbed...,observation:\n/testbed/doc/conf.py\r\n/testbed...,dvc render match py,if imagerenderer matches inner_id none if vega...,0,0,0
8,facelessuser__soupsieve.a8080d97.func_pm_remov...,facelessuser__soupsieve.a8080d97.func_pm_remov...,facelessuser__soupsieve,claude-3-7-sonnet-20250219,0,0,1181,22,0,1,...,<uploaded_files>\n/testbed\n</uploaded_files>\...,i'll help you implement the necessary changes ...,i'll help you implement the necessary changes ...,observation:\n/testbed/hatch_build.py\r\n/test...,observation:\n/testbed/hatch_build.py\r\n/test...,ptyprocess ptyprocess py,first check if echo is already off if not self...,0,0,0


In [5]:
TESTMETA_PATH = Path("/content/drive/MyDrive/fast-failure-detection/results/tool_testmeta_features_traj_level.csv")

testmeta_df = pd.read_csv(TESTMETA_PATH)
print(testmeta_df.shape)
testmeta_df.head()

(17190, 27)


,instance_id,traj_id,resolved,repo_id,repo,n_ftp,n_ptp,n_total_gold_tests,ftp_ptp_ratio,ftp_file_count,...,log1p_n_ftp,log1p_n_ptp,log1p_n_total_gold_tests,log1p_ftp_ptp_ratio,log1p_ftp_file_count,log1p_ptp_file_count,log1p_ftp_class_count,log1p_ptp_class_count,log1p_ftp_tests_per_file,log1p_ptp_tests_per_file
0,marshmallow-code__apispec.8b421526.func_pm_rem...,marshmallow-code__apispec.8b421526.func_pm_rem...,True,marshmallow-code__apispec,swesmith/marshmallow-code__apispec.8b421526,2,567,569,0.003527,1,...,1.098612,6.342121,6.345636,0.003521,0.693147,2.079442,0.693147,3.433987,1.098612,4.406719
1,marshmallow-code__marshmallow.9716fc62.func_pm...,marshmallow-code__marshmallow.9716fc62.func_pm...,True,marshmallow-code__marshmallow,swesmith/marshmallow-code__marshmallow.9716fc62,5,1183,1188,0.004227,1,...,1.791759,7.076654,7.080868,0.004218,0.693147,2.564949,1.098612,3.737670,1.791759,4.600995
2,sloria__environs.73c372df.func_pm_remove_assig...,sloria__environs.73c372df.func_pm_remove_assig...,True,sloria__environs,swesmith/sloria__environs.73c372df,3,102,105,0.029412,1,...,1.386294,4.634729,4.663439,0.028988,0.693147,0.693147,0.693147,2.484907,1.386294,4.634729
3,amueller__word_cloud.ec24191c.func_pm_ctrl_shu...,amueller__word_cloud.ec24191c.func_pm_ctrl_shu...,True,amueller__word_cloud,swesmith/amueller__word_cloud.ec24191c,2,71,73,0.028169,1,...,1.098612,4.276666,4.304065,0.027780,0.693147,1.098612,0.000000,0.000000,1.098612,3.597312
4,facelessuser__soupsieve.a8080d97.func_pm_remov...,facelessuser__soupsieve.a8080d97.func_pm_remov...,False,facelessuser__soupsieve,swesmith/facelessuser__soupsieve.a8080d97,5,376,381,0.013298,2,...,1.791759,5.932245,5.945421,0.013210,1.098612,4.418841,0.693147,4.356709,1.252763,1.720150


In [6]:
print(testmeta_df.columns.tolist())

['instance_id', 'traj_id', 'resolved', 'repo_id', 'repo', 'n_ftp', 'n_ptp', 'n_total_gold_tests', 'ftp_ptp_ratio', 'ftp_file_count', 'ptp_file_count', 'ftp_class_count', 'ptp_class_count', 'ftp_tests_per_file', 'ptp_tests_per_file', 'ftp_single_file', 'ptp_single_file', 'log1p_n_ftp', 'log1p_n_ptp', 'log1p_n_total_gold_tests', 'log1p_ftp_ptp_ratio', 'log1p_ftp_file_count', 'log1p_ptp_file_count', 'log1p_ftp_class_count', 'log1p_ptp_class_count', 'log1p_ftp_tests_per_file', 'log1p_ptp_tests_per_file']


In [8]:
testmeta_features = [
    "log1p_n_ftp",
    "log1p_n_ptp",
    "log1p_ftp_ptp_ratio",
    "log1p_ftp_file_count",
    "log1p_ptp_file_count",
]

merge_cols = ["traj_id"] + testmeta_features
df = baseline_df.merge(
    testmeta_df[merge_cols],
    on="traj_id",
    how="left"
)

print(df.shape)
print(df[testmeta_features].isna().mean())

(18156, 50)
log1p_n_ftp             0.053206
log1p_n_ptp             0.053206
log1p_ftp_ptp_ratio     0.094790
log1p_ftp_file_count    0.053206
log1p_ptp_file_count    0.053206
dtype: float64


In [10]:
baseline_features = [
    "patch_char_len",
    "num_lines_added",
    "num_lines_removed",
    "num_files_touched",
    "test_file_modified",
    "num_test_files_touched",
    "num_python_files_touched",
    "touch_requirements_file",
    "touch_config_file",
    "touch_docs_file",
    "touch_init_file",
    "touch_multiple_dirs",
    "contains_assert_change",
    "contains_import_change",
    "contains_exception_handling_change",
    "num_messages",
    "messages_char_len",
    "num_action_messages",
    "num_observation_messages",
    "num_submit_calls",
    "num_bash_calls",
    "num_edit_calls",
]

testmeta_features = [
    "log1p_n_ftp",
    "log1p_n_ptp",
    "log1p_ftp_ptp_ratio",
    "log1p_ftp_file_count",
    "log1p_ptp_file_count",
]

In [11]:
experiments = [
    ("baseline_only", baseline_features, True),
    ("testmeta_only", testmeta_features, False),
    ("baseline_plus_testmeta", baseline_features + testmeta_features, True),
]

In [6]:
all_results = []
all_summaries = []
all_thresholds = []
all_fixed_thresholds = []

for exp_name, feature_cols, use_tfidf in experiments:
    print(f"\n===== Running {exp_name} =====")
    print("num features:", len(feature_cols))
    print("use_tfidf:", use_tfidf)

    results_df, summary_df, threshold_selection_df, fixed_threshold_summary_df = run_cv(
        df=df,
        feature_cols=feature_cols,
        log_features=LOG_FEATURES,
        model_name=MODEL_NAME,
        use_message_tfidf=use_tfidf,
    )

    results_df = results_df.copy()
    results_df.insert(0, "experiment", exp_name)

    summary_df = summary_df.copy()
    summary_df.insert(0, "experiment", exp_name)

    threshold_selection_df = threshold_selection_df.copy()
    threshold_selection_df.insert(0, "experiment", exp_name)

    fixed_threshold_summary_df = fixed_threshold_summary_df.copy()
    fixed_threshold_summary_df.insert(0, "experiment", exp_name)

    all_results.append(results_df)
    all_summaries.append(summary_df)
    all_thresholds.append(threshold_selection_df)
    all_fixed_thresholds.append(fixed_threshold_summary_df)


===== Running baseline_plus_level1 =====
num features: 28
use_tfidf: True


KeyboardInterrupt: 

In [13]:
final_results = pd.concat(all_results, ignore_index=True)
final_summaries = pd.concat(all_summaries, ignore_index=True)
final_thresholds = pd.concat(all_thresholds, ignore_index=True)
final_fixed_thresholds = pd.concat(all_fixed_thresholds, ignore_index=True)

display_cols = [
    "experiment",
    "chosen_threshold",
    "success_precision",
    "success_recall",
    "success_f1",
    "failure_precision",
    "failure_recall",
    "failure_f1",
    "accuracy",
    "success_auc",
]

final_summaries[display_cols].sort_values("failure_f1", ascending=False)

,experiment,chosen_threshold,success_precision,success_recall,success_f1,failure_precision,failure_recall,failure_f1,accuracy,success_auc
2,baseline_plus_testmeta,0.3,0.531472,0.810999,0.637200,0.815240,0.539137,0.646665,0.645578,0.760253
0,baseline_only,0.3,0.512731,0.793856,0.619196,0.793988,0.513383,0.622146,0.624925,0.740781
1,testmeta_only,0.3,0.422141,0.779269,0.541100,0.690917,0.309628,0.420374,0.491095,0.570416


In [14]:
OUT_DIR = Path("/content/drive/MyDrive/fast-failure-detection/results/testmeta_ablation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

final_results.to_csv(OUT_DIR / "ablation_results.csv", index=False)
final_summaries.to_csv(OUT_DIR / "ablation_summary.csv", index=False)
final_thresholds.to_csv(OUT_DIR / "ablation_thresholds.csv", index=False)
final_fixed_thresholds.to_csv(OUT_DIR / "ablation_fixed_thresholds.csv", index=False)

print("Saved to:", OUT_DIR)

Saved to: /content/drive/MyDrive/fast-failure-detection/results/testmeta_ablation


In [3]:
baseline_df = pd.read_csv("/content/drive/MyDrive/fast-failure-detection/results/tool_features_full.csv")
baseline_df = baseline_df[baseline_df["patch_is_empty"] == 0].copy()

testmeta_df = pd.read_csv("/content/drive/MyDrive/fast-failure-detection/results/tool_testmeta_features_traj_level.csv")
align_df = pd.read_csv("/content/drive/MyDrive/fast-failure-detection/results/tool_alignment_features_traj_level.csv")

df = baseline_df.merge(testmeta_df, on=["traj_id", "instance_id", "resolved", "repo_id"], how="left")
df = df.merge(align_df, on=["traj_id", "instance_id", "resolved", "repo_id"], how="left")
print(df.shape)

(18156, 101)


In [4]:
baseline_features = [
    "patch_char_len",
    "num_lines_added",
    "num_lines_removed",
    "num_files_touched",
    "test_file_modified",
    "num_test_files_touched",
    "num_python_files_touched",
    "touch_requirements_file",
    "touch_config_file",
    "touch_docs_file",
    "touch_init_file",
    "touch_multiple_dirs",
    "contains_assert_change",
    "contains_import_change",
    "contains_exception_handling_change",
    "num_messages",
    "messages_char_len",
    "num_action_messages",
    "num_observation_messages",
    "num_submit_calls",
    "num_bash_calls",
    "num_edit_calls",
]

testmeta_features = [
    "log1p_n_ftp",
    "log1p_n_ptp",
    "log1p_ftp_ptp_ratio",
    "log1p_ftp_file_count",
    "log1p_ptp_file_count",
]

level1_features = [
    "has_ftp_file_overlap",
    "has_ptp_file_overlap",
    "ftp_file_overlap_count",
    "ptp_file_overlap_count",
    "ftp_file_overlap_ratio",
    "ptp_file_overlap_ratio",
]

level2_features = [
    "has_ftp_class_overlap",
    "has_ptp_class_overlap",
    "has_ftp_test_overlap",
    "has_ptp_test_overlap",
    "ftp_class_overlap_count",
    "ptp_class_overlap_count",
    "ftp_test_overlap_count",
    "ptp_test_overlap_count",
    "ftp_class_overlap_ratio",
    "ptp_class_overlap_ratio",
    "ftp_test_overlap_ratio",
    "ptp_test_overlap_ratio",
]

level3_features = [
    "ftp_first_file_hit_step_norm",
    "ptp_first_file_hit_step_norm",
    "ftp_first_class_hit_step_norm",
    "ptp_first_class_hit_step_norm",
    "ftp_first_test_hit_step_norm",
    "ptp_first_test_hit_step_norm",
]

In [10]:
df = df.copy()
df["resolved"] = pd.to_numeric(df["resolved"], errors="raise").astype(int)

print(df["resolved"].dtype)
print(df["resolved"].value_counts(dropna=False))

int64
resolved
0    11011
1     7145
Name: count, dtype: int64


In [11]:
all_results = []
all_summaries = []
all_thresholds = []
all_fixed_thresholds = []

for exp_name, feature_cols, use_tfidf in experiments:
    print(f"\n===== Running {exp_name} =====")
    print("num features:", len(feature_cols))
    print("use_tfidf:", use_tfidf)

    results_df, summary_df, threshold_selection_df, fixed_threshold_summary_df = run_cv(
        df=df,
        feature_cols=feature_cols,
        log_features=LOG_FEATURES,
        model_name=MODEL_NAME,
        use_message_tfidf=use_tfidf,
    )

    results_df = results_df.copy()
    results_df.insert(0, "experiment", exp_name)

    summary_df = summary_df.copy()
    summary_df.insert(0, "experiment", exp_name)

    threshold_selection_df = threshold_selection_df.copy()
    threshold_selection_df.insert(0, "experiment", exp_name)

    fixed_threshold_summary_df = fixed_threshold_summary_df.copy()
    fixed_threshold_summary_df.insert(0, "experiment", exp_name)

    all_results.append(results_df)
    all_summaries.append(summary_df)
    all_thresholds.append(threshold_selection_df)
    all_fixed_thresholds.append(fixed_threshold_summary_df)


===== Running baseline_plus_level1 =====
num features: 28
use_tfidf: True

[TF-IDF:assistant] top 30 features by mean TF-IDF:
parameter 0.063629
parameter parameter 0.048006
command 0.047395
testbed 0.047382
parameter function 0.046599
parameter command 0.046586
py parameter 0.04486
print 0.044774
method 0.042837
path 0.042416
function let 0.039816
parameter path 0.039695
function str_replace_editor 0.039695
str_replace_editor parameter 0.039695
str_replace_editor 0.039695
path testbed 0.039689
self 0.039408
class 0.039165
bash 0.036909
bash parameter 0.036862
function bash 0.036862
import 0.036075
edge 0.035994
return 0.035896
cases 0.035552
python 0.03518
try 0.034146
case 0.034035
edge cases 0.03212
def 0.03202

[TF-IDF:file_path] top 30 features by mean TF-IDF:
src 0.048171
reproduce 0.040538
py reproduce 0.032104
reproduce py 0.028063
init py 0.025426
init 0.025318
test 0.023957
core 0.021978
moto 0.0198
utils 0.016741
error 0.01667
py test 0.016558
dvc 0.016167
error py 0.016122

In [12]:
final_results = pd.concat(all_results, ignore_index=True)
final_summaries = pd.concat(all_summaries, ignore_index=True)
final_thresholds = pd.concat(all_thresholds, ignore_index=True)
final_fixed_thresholds = pd.concat(all_fixed_thresholds, ignore_index=True)

display_cols = [
    "experiment",
    "chosen_threshold",
    "success_precision",
    "success_recall",
    "success_f1",
    "failure_precision",
    "failure_recall",
    "failure_f1",
    "accuracy",
    "success_auc",
]

final_summaries[display_cols].sort_values("failure_f1", ascending=False)

,experiment,chosen_threshold,success_precision,success_recall,success_f1,failure_precision,failure_recall,failure_f1,accuracy,success_auc
1,baseline_plus_level1_level2,0.3,0.523932,0.790313,0.625996,0.797283,0.538223,0.641242,0.637477,0.750611
2,baseline_plus_all_alignment,0.3,0.522357,0.789609,0.624947,0.795934,0.535673,0.639205,0.636211,0.750810
0,baseline_plus_level1,0.3,0.517894,0.788226,0.620748,0.792669,0.527585,0.631825,0.630210,0.746572


In [13]:
OUT_DIR = Path("/content/drive/MyDrive/fast-failure-detection/results/testmeta_ablation")
OUT_DIR.mkdir(parents=True, exist_ok=True)

final_results.to_csv(OUT_DIR / "ablation_results.csv", index=False)
final_summaries.to_csv(OUT_DIR / "ablation_summary.csv", index=False)
final_thresholds.to_csv(OUT_DIR / "ablation_thresholds.csv", index=False)
final_fixed_thresholds.to_csv(OUT_DIR / "ablation_fixed_thresholds.csv", index=False)

print("Saved to:", OUT_DIR)

Saved to: /content/drive/MyDrive/fast-failure-detection/results/testmeta_ablation
